# Figure 6 — UCB: optimism with a dial

The same posterior scored with beta = 0.5, 2 and 5, built up one beta at a time for a click-build (same figsize/xlim/ylim recipe as fig. 5's PI/EI/UCB build). The argmax walks from exploitation to exploration as beta grows. beta = 2 is a one-sided ~98% bound, not 95%. No title -- each row's own label is beta's value.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "_shared"))
sys.path.insert(0, str(pathlib.Path.cwd() / "_shared"))
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import style, gp as gpmod, landscape as land, doe as doemod
style.use_deck_style()
OUT = "../lecture_12_figures/generated"

In [ ]:

xs = np.linspace(0, 10, 700)
OX = np.array([1.15, 2.90, 4.30, 6.10, 9.30])
OY = land.f1d(OX)
g = gpmod.GP(gpmod.matern52, ls=0.85, sf=1.0, sn=0.03).fit(OX[:, None], OY)
mu, sd = g.predict(xs[:, None])

COLB = {0.5: style.INK, 2.0: style.BLUE, 5.0: style.GOLD}
A_UCB = {b: gpmod.ucb(mu, sd, b) for b in COLB}


def draw(betas, fname):
    # same figsize/xlim/ylim recipe as fig_05's draw(): n rows -> taller figure,
    # top panel pinned to fig_05's own (0, 10) x (-1.9, 2.5) box
    n = 1 + len(betas)
    fig, axes = plt.subplots(n, 1, figsize=(style.FIG_W_FULL, 1.35 + 1.15 * n),
                             sharex=True, gridspec_kw=dict(hspace=0.20,
                             height_ratios=[2.1] + [1.0] * len(betas)))
    ax = axes[0]
    ax.fill_between(xs, mu - 2 * sd, mu + 2 * sd, color=style.TEAL, alpha=0.16, lw=0)
    ax.plot(xs, mu, color=style.TEAL, lw=2.0)
    ax.plot(OX, OY, "o", ms=6.5, color=style.RED, mec="white", mew=1.0, zorder=6)
    style.ylabel(ax, "yield (arb.)")
    ax.set_xlim(0, 10)
    ax.set_ylim(-1.9, 2.5)

    for axb, b in zip(axes[1:], betas):
        c = COLB[b]
        a = A_UCB[b]
        axb.plot(xs, a, color=c, lw=1.8)
        xstar = xs[int(np.argmax(a))]
        axb.axvline(xstar, color=c, lw=1.0, ls="--")
        ax.axvline(xstar, color=c, lw=1.0, ls="--", alpha=0.75)
        axb.plot([xstar], [a.max()], "v", ms=8, color=c)
        style.ylabel(axb, fr"$\beta$={b}", color=c, fontweight="bold")
        axb.set_yticks([])
        axb.set_xlim(0, 10)
    style.xlabel(axes[-1], "reaction parameter  x")
    style.save(fig, fname, OUT)
    plt.close(fig)


draw([0.5], "fig_06a_ucb_beta_0p5")
draw([0.5, 2.0], "fig_06b_ucb_beta_0p5_2p0")
draw([0.5, 2.0, 5.0], "fig_06_ucb_beta_sweep")